# Agent 2 — AI Summarization

Reads all reconciliation runs with ai_enrichment_status = PENDING,
assembles a cross-layer evidence bundle for each run,
calls the AI model, and writes the enrichment back.

Evidence bundle includes:
  - Which reconciliation checks failed and by how much
  - Which layers are involved (Raw, Silver, Gold)
  - Agent 1 context (what was found at the raw layer)
  - Baseline survival rates for comparison

Reads from:
  uc2_reconciliation_run_b
  uc2_reconciliation_results_b
  uc2_layer_registry_b
  uc1_validation_run_b      (Agent 1 context)
  uc1_validation_results_b  (Agent 1 check details)

Updates:
  uc2_reconciliation_results_b
  uc2_reconciliation_run_b


In [0]:
import requests
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

RECONCILIATION_RUN     = f"{CATALOG}.{SCHEMA}.uc2_reconciliation_run_b"
RECONCILIATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc2_reconciliation_results_b"
LAYER_REGISTRY         = f"{CATALOG}.{SCHEMA}.uc2_layer_registry_b"
AGENT1_RUN             = f"{CATALOG}.{SCHEMA}.uc1_validation_run_b"
AGENT1_RESULTS         = f"{CATALOG}.{SCHEMA}.uc1_validation_results_b"

# ── Model Configuration ────────────────────────────────────────────────────

MODEL_ENDPOINT = "databricks-gpt-oss-20b"
MODEL_VERSION  = "gpt-oss-20b-080525"
MAX_TOKENS     = 4096
TEMPERATURE    = 0.1

# ── Workspace Connection ───────────────────────────────────────────────────

WORKSPACE_HOST  = spark.conf.get("spark.databricks.workspaceUrl")
WORKSPACE_TOKEN = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

HEADERS = {
    "Authorization": f"Bearer {WORKSPACE_TOKEN}",
    "Content-Type" : "application/json"
}

MODEL_URL = (
    f"https://{WORKSPACE_HOST}"
    f"/serving-endpoints/{MODEL_ENDPOINT}/invocations"
)

VALID_FAILURE_CATEGORIES = {
    "SCHEMA_ISSUE",
    "VOLUME_ISSUE",
    "QUALITY_ISSUE",
    "PIPELINE_ISSUE",
    "COMBINED_ISSUE"
}

print("=" * 60)
print("AGENT 2 — AI SUMMARIZATION")
print("=" * 60)
print(f"Model endpoint : {MODEL_ENDPOINT}")
print(f"Model URL      : {MODEL_URL}")
print(f"Token length   : {len(WORKSPACE_TOKEN)}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

## Step 1: Load Pending Reconciliation Runs

In [0]:
def load_pending_runs() -> list:
    """
    Loads all reconciliation runs where AI enrichment is PENDING.
    """
    df = (
        spark.table(RECONCILIATION_RUN)
             .filter(F.col("ai_enrichment_status") == "PENDING")
             .orderBy("business_date", "source_fully_qualified_name")
    )

    rows = [row.asDict() for row in df.collect()]

    print(f"Pending runs found: {len(rows)}")
    for r in rows:
        print(
            f"  {r['business_date']}  "
            f"{r['source_fully_qualified_name'].split('.')[-1]:<35}  "
            f"Status: {r['overall_status']}  "
            f"Critical: {r['total_checks_critical']}"
        )

    return rows


pending_runs = load_pending_runs()

if not pending_runs:
    print("\nNo pending runs. Nothing to process.")
    NO_PENDING_RUNS = True
else:
    NO_PENDING_RUNS = False
    print(f"\nProceeding to process {len(pending_runs)} run(s).")

## Step 2: Evidence Bundle Assembly Functions

In [0]:
def load_failed_checks(reconciliation_run_id: str) -> list:
    """
    Loads all WARNING and CRITICAL reconciliation checks
    for a given run.
    """
    df = (
        spark.table(RECONCILIATION_RESULTS)
             .filter(
                 (F.col("reconciliation_run_id") == reconciliation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .otherwise(2),
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]


def load_agent1_detail(
    source_fqn    : str,
    business_date : str
) -> dict:
    """
    Loads Agent 1 run summary and failed check details
    for the same source table and business date.

    This gives Agent 2 AI the context of what Agent 1
    found at the raw layer before reconciliation issues appeared.
    """
    try:
        # Get run summary
        run_rows = (
            spark.table(AGENT1_RUN)
                 .filter(
                     (F.col("fully_qualified_name") == source_fqn) &
                     (F.col("business_date") == F.lit(str(business_date)))
                 )
                 .orderBy(F.col("created_timestamp").desc())
                 .limit(1)
                 .collect()
        )

        if not run_rows:
            return {"available": False}

        run = run_rows[0].asDict()

        # Get failed check summaries from Agent 1
        agent1_failed = (
            spark.table(AGENT1_RESULTS)
                 .filter(
                     (F.col("validation_run_id") == run["validation_run_id"]) &
                     (F.col("check_status").isin(["WARNING", "CRITICAL"]))
                 )
                 .select(
                     "check_type",
                     "column_name",
                     "check_status",
                     "observed_value",
                     "baseline_value",
                     "deviation_percentage"
                 )
                 .collect()
        )

        failed_checks = [row.asDict() for row in agent1_failed]

        return {
            "available"          : True,
            "overall_status"     : run.get("overall_status"),
            "total_checks_run"   : run.get("total_checks_run"),
            "total_checks_critical": run.get("total_checks_critical"),
            "total_checks_warned": run.get("total_checks_warned"),
            "ai_enrichment_status": run.get("ai_enrichment_status"),
            "failed_checks"      : failed_checks
        }

    except Exception as e:
        logger.warning(f"Could not load Agent 1 detail: {str(e)}")
        return {"available": False}


def load_layer_context(source_fqn: str) -> dict:
    """
    Loads layer registry metadata for the source table.
    """
    try:
        rows = (
            spark.table(LAYER_REGISTRY)
                 .filter(
                     (F.col("source_fully_qualified_name") == source_fqn) &
                     (F.col("is_active") == True)
                 )
                 .orderBy("layer_name")
                 .collect()
        )

        layers = {}
        for row in rows:
            r = row.asDict()
            layers[r["layer_name"]] = {
                "table"              : r["layer_fully_qualified_name"],
                "expected_survival"  : r.get("expected_survival_rate"),
                "expected_latency"   : r.get("expected_latency_minutes"),
                "metrics"            : r.get("reconciliation_metrics")
            }
        return layers

    except Exception as e:
        logger.warning(f"Could not load layer context: {str(e)}")
        return {}


def assemble_evidence_bundle(
    run_record    : dict,
    failed_checks : list,
    agent1_detail : dict,
    layer_context : dict
) -> dict:
    """
    Assembles the complete evidence bundle for Agent 2 AI summarization.

    Includes:
    - Reconciliation run context (which table, which date)
    - Failed cross-layer checks with observed vs expected values
    - Layer configuration (expected survival rates)
    - Agent 1 findings at the raw layer (what was already known)
    """
    fqn = run_record["source_fully_qualified_name"]

    # Parse check_detail JSON for each failed check
    cleaned_checks = []
    for check in failed_checks:
        detail = {}
        if check.get("check_detail"):
            try:
                detail = json.loads(check["check_detail"])
            except (json.JSONDecodeError, TypeError):
                detail = {}

        cleaned_checks.append({
            "from_layer"        : check["from_layer"],
            "to_layer"          : check["to_layer"],
            "check_type"        : check["check_type"],
            "metric_name"       : check["metric_name"],
            "check_status"      : check["check_status"],
            "source_value"      : check["source_value"],
            "target_value"      : check["target_value"],
            "expected_value"    : check["expected_value"],
            "deviation_pct"     : check["deviation_percentage"],
            "threshold_warning" : check["threshold_warning"],
            "threshold_critical": check["threshold_critical"],
            "detail"            : detail
        })

    return {
        "context": {
            "reconciliation_run_id"      : run_record["reconciliation_run_id"],
            "source_fully_qualified_name": fqn,
            "table_name"                 : fqn.split(".")[-1],
            "business_date"              : str(run_record["business_date"]),
            "overall_status"             : run_record["overall_status"],
            "total_checks_run"           : run_record["total_checks_run"],
            "total_checks_critical"      : run_record["total_checks_critical"],
            "total_checks_warned"        : run_record["total_checks_warned"]
        },
        "failed_reconciliation_checks": cleaned_checks,
        "layer_configuration"         : layer_context,
        "agent1_context"              : agent1_detail
    }


print("Evidence bundle functions loaded.")

## Step 3: Prompt Builder

In [0]:
def build_prompt(evidence_bundle: dict) -> str:
    """
    Builds the AI prompt for cross-layer reconciliation analysis.

    The prompt provides:
    - Which table and date are being analyzed
    - What reconciliation checks failed between which layers
    - What Agent 1 found at the raw layer (if anything)
    - Layer configuration for context
    """
    ctx           = evidence_bundle["context"]
    table_name    = ctx["table_name"]
    business_date = ctx["business_date"]
    overall       = ctx["overall_status"]
    failed_checks = evidence_bundle["failed_reconciliation_checks"]
    layer_config  = evidence_bundle["layer_configuration"]
    agent1        = evidence_bundle["agent1_context"]

    # Format failed reconciliation checks
    checks_text = ""
    for i, check in enumerate(failed_checks, 1):
        checks_text += f"""
  Check {i}:
    From layer    : {check['from_layer']}
    To layer      : {check['to_layer']}
    Check type    : {check['check_type']}
    Metric        : {check['metric_name']}
    Status        : {check['check_status']}
    Source value  : {check['source_value']}
    Target value  : {check['target_value']}
    Expected value: {check['expected_value']}
    Deviation     : {check['deviation_pct']}%
    Warning at    : {check['threshold_warning']}%
    Critical at   : {check['threshold_critical']}%
    Detail        : {json.dumps(check['detail'], indent=6)}
"""

    # Format layer configuration
    layer_text = ""
    for layer_name, config in layer_config.items():
        layer_text += f"""
  {layer_name}:
    Table            : {config.get('table')}
    Expected survival: {config.get('expected_survival')}
    Expected latency : {config.get('expected_latency')} minutes
    Metrics tracked  : {config.get('metrics')}
"""

    # Format Agent 1 context
    if agent1.get("available"):
        agent1_failed = agent1.get("failed_checks", [])
        agent1_text = f"""
  Agent 1 overall status   : {agent1.get('overall_status')}
  Agent 1 critical checks  : {agent1.get('total_checks_critical')}
  Agent 1 warning checks   : {agent1.get('total_checks_warned')}
  Agent 1 AI status        : {agent1.get('ai_enrichment_status')}

  Agent 1 failed checks at raw layer:
"""
        if agent1_failed:
            for fc in agent1_failed:
                agent1_text += (
                    f"    {fc['check_type']} on {fc['column_name'] or 'table-level'}: "
                    f"{fc['check_status']} "
                    f"(observed {fc['observed_value']}, "
                    f"deviation {fc['deviation_percentage']}%)\n"
                )
        else:
            agent1_text += "    No raw layer failures detected by Agent 1.\n"
    else:
        agent1_text = "  Agent 1 context not available for this table and date."

    prompt = f"""You are a senior data engineer analyzing cross-layer
data reconciliation results for a retail data platform.

A reconciliation agent has detected discrepancies between
data layers (Raw → Silver → Gold) for a specific table
and business date.

CRITICAL SCOPE CONSTRAINT:
You are analyzing ONE table only: {table_name}
Business date: {business_date}
Do not reference any other table in your response.

TABLE AND DATE CONTEXT:
  Table name        : {table_name}
  Business date     : {business_date}
  Overall status    : {overall}
  Failed checks     : {len(failed_checks)}

FAILED RECONCILIATION CHECKS:
({len(failed_checks)} check(s) failed between data layers)
{checks_text}

LAYER CONFIGURATION:
(Expected survival rates and latency between layers)
{layer_text}

AGENT 1 RAW LAYER CONTEXT:
(What Agent 1 found when it validated the raw layer
 for this same table and business date)
{agent1_text}

INSTRUCTIONS:
Analyze the cross-layer discrepancies and produce a JSON
response. Every sentence must reference {table_name} and
{business_date} only.

Key questions to answer:
1. Which layer is the origin of the problem (Raw, Silver, or Gold)?
2. Is this consistent with what Agent 1 found at the raw layer?
3. What is the most likely root cause of the cross-layer drift?

{{
  "ai_summary": "2-3 sentences explaining what cross-layer
                  discrepancy was detected in {table_name}
                  on {business_date}. Name the specific layers,
                  checks, and metrics affected. Reference Agent 1
                  findings if relevant.",

  "ai_recommended_action": "1-2 specific actionable steps.
                             Reference {table_name}, the specific
                             layer where the problem originates,
                             and {business_date}.",

  "ai_failure_category": "One of: SCHEMA_ISSUE, VOLUME_ISSUE,
                           QUALITY_ISSUE, PIPELINE_ISSUE,
                           COMBINED_ISSUE",

  "ai_confidence_score": 0.0 to 1.0
}}

Respond with valid JSON only. No markdown."""

    return prompt


print("Prompt builder loaded.")

## Step 4: AI Model Caller

In [0]:
def extract_content_from_response(response_data: dict) -> str:
    """
    Extracts text content from model response.
    Handles the gpt-oss-20b reasoning block format.
    """
    choices = response_data.get("choices", [])
    if not choices:
        return ""

    first_choice = choices[0]
    message      = first_choice.get("message", {})
    content      = message.get("content", "")

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts     = []
        reasoning_text = []

        for block in content:
            if not isinstance(block, dict):
                if isinstance(block, str):
                    text_parts.append(block)
                continue

            block_type = block.get("type", "")

            if block_type == "text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)
            elif block_type == "output_text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)
            elif block_type == "reasoning":
                summary = block.get("summary", [])
                for s in summary:
                    if isinstance(s, dict):
                        s_text = s.get("text", "")
                        if s_text:
                            reasoning_text.append(s_text)

        if text_parts:
            return " ".join(text_parts)
        if reasoning_text:
            return " ".join(reasoning_text)

    text = first_choice.get("text", "")
    if text:
        return text

    return ""


def call_ai_model(prompt: str, invocation_id: str) -> dict:
    """
    Calls the AI model and returns parsed response.
    """
    payload = {
        "messages": [
            {
                "role"   : "system",
                "content": (
                    "You are a data quality engineer for a retail "
                    "data platform. Always respond with valid JSON "
                    "only. Never include markdown or explanation "
                    "outside the JSON object."
                )
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        "max_tokens" : MAX_TOKENS,
        "temperature": TEMPERATURE
    }

    try:
        response = requests.post(
            MODEL_URL,
            headers=HEADERS,
            json=payload,
            timeout=180
        )

        if response.status_code != 200:
            return {
                "success"     : False,
                "error"       : f"HTTP {response.status_code}: {response.text[:500]}",
                "raw_response": response.text
            }

        response_data = response.json()
        raw_content   = extract_content_from_response(response_data)

        if not isinstance(raw_content, str):
            raw_content = str(raw_content)

        if not raw_content.strip():
            return {
                "success"     : False,
                "error"       : "Empty content from model",
                "raw_response": str(response_data)
            }

        parsed = parse_ai_response(raw_content)

        if parsed["success"]:
            return {
                "success"              : True,
                "ai_summary"           : parsed["ai_summary"],
                "ai_recommended_action": parsed["ai_recommended_action"],
                "ai_failure_category"  : parsed["ai_failure_category"],
                "ai_confidence_score"  : parsed["ai_confidence_score"],
                "raw_response"         : raw_content
            }
        else:
            return {
                "success"     : False,
                "error"       : parsed["error"],
                "raw_response": raw_content
            }

    except requests.exceptions.Timeout:
        return {
            "success"     : False,
            "error"       : "Model call timed out after 180 seconds",
            "raw_response": None
        }
    except Exception as e:
        logger.error(f"Unexpected error: {str(e)}")
        return {
            "success"     : False,
            "error"       : str(e),
            "raw_response": None
        }


def parse_ai_response(raw_content: str) -> dict:
    """
    Parses raw model response into structured fields.
    """
    if not raw_content or not raw_content.strip():
        return {"success": False, "error": "Empty response"}

    content = raw_content.strip()

    if "```json" in content:
        content = content.split("```json")[1]
        if "```" in content:
            content = content.split("```")[0]
    elif "```" in content:
        parts = content.split("```")
        if len(parts) >= 2:
            content = parts[1]

    content = content.strip()
    start   = content.find("{")
    end     = content.rfind("}") + 1

    if start == -1 or end == 0:
        return {
            "success": False,
            "error"  : f"No JSON found. Preview: {content[:200]}"
        }

    try:
        parsed = json.loads(content[start:end])
    except json.JSONDecodeError as e:
        return {
            "success": False,
            "error"  : f"JSON parse error: {str(e)}"
        }

    required = [
        "ai_summary",
        "ai_recommended_action",
        "ai_failure_category",
        "ai_confidence_score"
    ]

    missing = [f for f in required if f not in parsed]
    if missing:
        return {
            "success": False,
            "error"  : f"Missing fields: {missing}"
        }

    category = str(parsed.get("ai_failure_category", "")).upper().strip()
    if category not in VALID_FAILURE_CATEGORIES:
        category = "COMBINED_ISSUE"

    try:
        confidence = float(parsed.get("ai_confidence_score", 0.5))
        confidence = max(0.0, min(1.0, confidence))
    except (TypeError, ValueError):
        confidence = 0.5

    return {
        "success"              : True,
        "ai_summary"           : str(parsed.get("ai_summary", "")).strip(),
        "ai_recommended_action": str(parsed.get("ai_recommended_action", "")).strip(),
        "ai_failure_category"  : category,
        "ai_confidence_score"  : confidence
    }


print("AI model caller loaded.")

## Step 5: Write AI Enrichment

In [0]:
def write_ai_enrichment(
    reconciliation_run_id  : str,
    ai_summary             : str,
    ai_recommended_action  : str,
    ai_failure_category    : str,
    ai_confidence_score    : float,
    ai_invocation_id       : str,
    enriched_timestamp     : datetime
) -> bool:
    """
    Updates all failed check rows for a reconciliation run
    with the AI enrichment fields.
    """
    try:
        safe_summary = ai_summary.replace("'", "\\'")
        safe_action  = ai_recommended_action.replace("'", "\\'")

        spark.sql(f"""
            UPDATE {RECONCILIATION_RESULTS}
            SET
                ai_summary              = '{safe_summary}',
                ai_recommended_action   = '{safe_action}',
                ai_failure_category     = '{ai_failure_category}',
                ai_confidence_score     = {ai_confidence_score},
                ai_model_version        = '{MODEL_VERSION}',
                ai_invocation_id        = '{ai_invocation_id}',
                ai_enriched_timestamp   = '{enriched_timestamp}'
            WHERE reconciliation_run_id = '{reconciliation_run_id}'
              AND check_status IN ('WARNING', 'CRITICAL')
        """)

        logger.info(f"AI enrichment written for run {reconciliation_run_id}")
        return True

    except Exception as e:
        logger.error(f"Failed to write AI enrichment: {str(e)}")
        return False


def update_run_status(
    reconciliation_run_id : str,
    ai_status             : str,
    enriched_timestamp    : datetime
) -> bool:
    """
    Updates the reconciliation run with AI enrichment status.
    """
    try:
        spark.sql(f"""
            UPDATE {RECONCILIATION_RUN}
            SET
                ai_enrichment_status    = '{ai_status}',
                ai_enrichment_timestamp = '{enriched_timestamp}'
            WHERE reconciliation_run_id = '{reconciliation_run_id}'
        """)
        return True

    except Exception as e:
        logger.error(f"Failed to update run status: {str(e)}")
        return False


print("Results writer loaded.")

## Step 6: Process All Pending Runs

In [0]:
runs_processed = 0
runs_succeeded = 0
runs_failed    = 0
invocation_log = []

if NO_PENDING_RUNS:
    print("Skipping — no pending runs found.")
else:
    print(f"Processing {len(pending_runs)} pending run(s)...\n")

    for run_record in pending_runs:

        run_id   = run_record["reconciliation_run_id"]
        fqn      = run_record["source_fully_qualified_name"]
        biz_date = run_record["business_date"]

        print(f"{'='*60}")
        print(f"Run ID    : {run_id}")
        print(f"Table     : {fqn}")
        print(f"Date      : {biz_date}")
        print(f"{'='*60}")

        invocation_id = str(uuid.uuid4())

        try:
            # ── Load failed checks ─────────────────────────────────────────
            failed_checks = load_failed_checks(run_id)
            print(f"Failed checks loaded   : {len(failed_checks)}")

            if not failed_checks:
                print("No failed checks found. Skipping AI call.")
                update_run_status(run_id, "NOT_REQUIRED", datetime.now())
                runs_processed += 1
                continue

            # ── Load Agent 1 context ───────────────────────────────────────
            agent1_detail = load_agent1_detail(fqn, str(biz_date))
            print(
                f"Agent 1 context        : "
                f"{'Available' if agent1_detail.get('available') else 'Not available'}"
            )
            if agent1_detail.get("available"):
                print(
                    f"Agent 1 status         : "
                    f"{agent1_detail.get('overall_status')} "
                    f"({agent1_detail.get('total_checks_critical')} critical)"
                )

            # ── Load layer context ─────────────────────────────────────────
            layer_context = load_layer_context(fqn)
            print(f"Layers loaded          : {list(layer_context.keys())}")

            # ── Assemble evidence bundle ───────────────────────────────────
            evidence_bundle = assemble_evidence_bundle(
                run_record,
                failed_checks,
                agent1_detail,
                layer_context
            )
            print(f"Evidence bundle assembled.")

            # ── Build prompt ───────────────────────────────────────────────
            prompt = build_prompt(evidence_bundle)
            print(f"Prompt length          : {len(prompt)} characters")

            # ── Call AI model ──────────────────────────────────────────────
            print(f"Calling model          : {MODEL_ENDPOINT}")
            ai_result = call_ai_model(prompt, invocation_id)

            if ai_result["success"]:
                print(f"Model call succeeded.")
                print(f"Category               : {ai_result['ai_failure_category']}")
                print(f"Confidence             : {ai_result['ai_confidence_score']}")
                print(f"Summary preview        : {ai_result['ai_summary'][:100]}...")

                enriched_at   = datetime.now()
                write_success = write_ai_enrichment(
                    reconciliation_run_id  = run_id,
                    ai_summary             = ai_result["ai_summary"],
                    ai_recommended_action  = ai_result["ai_recommended_action"],
                    ai_failure_category    = ai_result["ai_failure_category"],
                    ai_confidence_score    = ai_result["ai_confidence_score"],
                    ai_invocation_id       = invocation_id,
                    enriched_timestamp     = enriched_at
                )

                if write_success:
                    update_run_status(run_id, "COMPLETED", enriched_at)
                    runs_succeeded += 1
                    print(f"Status                 : COMPLETED ✅")
                else:
                    update_run_status(run_id, "FAILED", datetime.now())
                    runs_failed += 1
                    print(f"Status                 : FAILED (write error) ❌")

            else:
                print(f"Model call failed: {ai_result.get('error')}")
                update_run_status(run_id, "FAILED", datetime.now())
                runs_failed += 1

            invocation_log.append({
                "run_id"       : run_id,
                "fqn"          : fqn,
                "business_date": str(biz_date),
                "invocation_id": invocation_id,
                "success"      : ai_result["success"],
                "category"     : ai_result.get("ai_failure_category"),
                "confidence"   : ai_result.get("ai_confidence_score"),
                "error"        : ai_result.get("error")
            })

        except Exception as e:
            print(f"UNEXPECTED ERROR: {str(e)}")
            import traceback
            traceback.print_exc()
            update_run_status(run_id, "FAILED", datetime.now())
            runs_failed += 1

        finally:
            runs_processed += 1
            print()

    print(f"Processing complete.")
    print(f"  Processed  : {runs_processed}")
    print(f"  Succeeded  : {runs_succeeded}")
    print(f"  Failed     : {runs_failed}")

## Step 7: Verify AI Enrichment Results

In [0]:
print("=" * 60)
print("AI ENRICHMENT VERIFICATION")
print("=" * 60)

# Run status
print("\n1. Reconciliation Run AI Status:\n")
spark.sql(f"""
    SELECT
        business_date,
        SUBSTRING_INDEX(source_fully_qualified_name, '.', -1) AS table_name,
        overall_status,
        ai_enrichment_status,
        ai_enrichment_timestamp
    FROM {RECONCILIATION_RUN}
    ORDER BY business_date, source_fully_qualified_name
""").show(20, truncate=False)

# AI enriched results
print("\n2. AI Enriched Results:\n")
spark.sql(f"""
    SELECT
        r.business_date,
        SUBSTRING_INDEX(r.source_fully_qualified_name, '.', -1) AS table_name,
        r.from_layer,
        r.to_layer,
        r.check_type,
        r.check_status,
        r.ai_failure_category,
        ROUND(r.ai_confidence_score, 2)                         AS confidence,
        r.ai_summary,
        r.ai_recommended_action
    FROM {RECONCILIATION_RESULTS} r
    WHERE r.ai_summary IS NOT NULL
    ORDER BY
        r.business_date,
        r.source_fully_qualified_name,
        r.check_type
""").show(50, truncate=False)

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 2 AI SUMMARIZATION COMPLETE")
print("=" * 60)

if NO_PENDING_RUNS:
    print("Runs processed   : 0 (no pending runs)")
else:
    print(f"Runs processed   : {runs_processed}")
    print(f"Runs succeeded   : {runs_succeeded}")
    print(f"Runs failed      : {runs_failed}")
    print()
    for entry in invocation_log:
        status = "✅" if entry["success"] else "❌"
        print(
            f"  {status} {entry['business_date']}  "
            f"{entry['fqn'].split('.')[-1]:<35}  "
            f"Category: {entry['category']}  "
            f"Confidence: {entry['confidence']}"
        )

print(f"\nModel used       : {MODEL_ENDPOINT}")
print(f"Completed at     : {datetime.now()}")
print("=" * 60)